# 36. Solar Prominences: Observations — Implementation / 구현

**Paper**: Parenti, S., *Living Rev. Solar Phys.* **11**, 1 (2014). [DOI: 10.12942/lrsp-2014-1]

## Purpose / 목적

**English.** This notebook implements and visualizes the key observational and theoretical concepts from Parenti's (2014) review of solar prominences, providing numerical intuition for:
1. Kippenhahn–Schlüter prominence support model (magnetic tension balancing gravity)
2. Synthetic Hα line profile from prominence plasma
3. Formation temperature vs density phase diagram for prominence plasma
4. Filament channel magnetic dip visualization (2D field lines)
5. Radiative cooling rate vs temperature for prominence plasma

**한국어.** 본 노트북은 Parenti (2014) 태양 프로미넌스 리뷰의 핵심 관측적·이론적 개념을 구현·시각화하여 수치적 직관을 제공한다:
1. Kippenhahn–Schlüter 프로미넌스 지지 모델 (자기 장력이 중력을 상쇄)
2. 프로미넌스 플라즈마의 Hα 선 프로파일 합성
3. 프로미넌스 플라즈마의 형성 온도–밀도 상태 다이어그램
4. 필라멘트 채널 자기 dip 시각화 (2D 자기력선)
5. 프로미넌스 플라즈마의 복사 냉각률–온도 곡선

## 0. Imports and Constants / 라이브러리와 상수

In [ ]:
"""Imports and physical constants in CGS units.

Google-style: set up NumPy/Matplotlib and define reusable physical
constants in CGS (centimeter-gram-second) system, which is standard
in solar physics literature.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Physical constants in CGS / CGS 단위 물리 상수
K_B = 1.380649e-16      # Boltzmann constant [erg/K]
M_H = 1.6735575e-24     # Hydrogen mass [g]
M_E = 9.1093837e-28     # Electron mass [g]
C_LIGHT = 2.99792458e10 # Speed of light [cm/s]
G_SUN = 2.74e4          # Solar surface gravity [cm/s^2]
R_SUN = 6.957e10        # Solar radius [cm]

# Prominence nominal values / 프로미넌스 대표값
T_PROM = 8.0e3          # Prominence core temperature [K]
N_PROM = 1.0e11         # Prominence core electron density [cm^-3]
B_PROM = 10.0           # Prominence magnetic field [G]
MU_PROM = 0.7           # Mean molecular weight (partially ionized H+He)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
print('Constants loaded. Prominence: T={:.0f} K, n={:.2e} cm^-3, B={} G'.format(
    T_PROM, N_PROM, B_PROM))

## 1. Kippenhahn–Schlüter Support Model / KS 지지 모델

**English.** The 1957 Kippenhahn–Schlüter model treats the prominence as an infinite thin vertical sheet supported against gravity by magnetic tension. The vertical force balance reduces to:

$$ B_x \frac{\partial B_z}{\partial x} = 4\pi \rho g $$

With isothermal stratification, the canonical solutions are:
- $B_z(x) = B_{y0} \tanh(x/H_p)$
- $\rho(x) = \rho_0 / \cosh^2(x/H_p)$

where $H_p = k_B T / (\mu m_H g)$ is the pressure scale height.

**한국어.** 1957년 KS 모델은 프로미넌스를 무한 얇은 수직 sheet로 다루며 자기 장력이 중력을 상쇄한다. 수직 힘 수지는 $B_x \partial B_z/\partial x = 4\pi \rho g$로 요약된다. 등온 적분 해는 $B_z(x) = B_{y0} \tanh(x/H_p)$, $\rho(x) = \rho_0 / \cosh^2(x/H_p)$이며, $H_p$는 압력 스케일 높이이다.

In [ ]:
def kippenhahn_schluter_profile(x_km, T=T_PROM, mu=MU_PROM, B_x=B_PROM, n_0=N_PROM):
    """Compute the Kippenhahn-Schluter equilibrium profile.

    Args:
        x_km: Horizontal coordinate across sheet [km].
        T: Temperature [K].
        mu: Mean molecular weight.
        B_x: Along-prominence field component [G].
        n_0: Central electron density [cm^-3].

    Returns:
        dict with pressure scale height H_p [km], B_z(x) [G], rho(x) [g/cm^3],
        and the transverse field amplitude B_y_0 [G].
    """
    H_p_cm = K_B * T / (mu * M_H * G_SUN)  # pressure scale height in cm
    H_p_km = H_p_cm / 1e5
    x_cm = x_km * 1e5
    rho_0 = n_0 * mu * M_H  # central mass density [g/cm^3]
    rho = rho_0 / np.cosh(x_cm / H_p_cm) ** 2
    # Canonical KS solution: B_z(x) = B_y_0 * tanh(x/H_p)
    # where B_y_0 provides the tension needed to support the plasma column.
    B_y_0 = np.sqrt(8 * np.pi * rho_0 * K_B * T / mu / M_H)
    B_z = B_y_0 * np.tanh(x_cm / H_p_cm)
    return {"H_p_km": H_p_km, "B_z": B_z, "rho": rho, "B_y_0": B_y_0}


# Compute profile and plot / 프로파일 계산과 플롯
x_km = np.linspace(-1500, 1500, 400)
profile = kippenhahn_schluter_profile(x_km)
print(f"Pressure scale height H_p = {profile['H_p_km']:.1f} km")
print(f"Transverse field amplitude B_y_0 = {profile['B_y_0']:.2f} G")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_km, profile['rho'] / (MU_PROM * M_H), 'b-', lw=2)
axes[0].set_xlabel('x [km] (horizontal across sheet)')
axes[0].set_ylabel('n [cm$^{-3}$]')
axes[0].set_title('KS density profile\n밀도 프로파일')
axes[0].grid(alpha=0.3)

axes[1].plot(x_km, profile['B_z'], 'r-', lw=2)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_xlabel('x [km]')
axes[1].set_ylabel('B$_z$ [G]')
axes[1].set_title('KS vertical field profile\n수직 자장 프로파일')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Synthetic Hα Line Profile from Prominence / Hα 선 프로파일 합성

**English.** The Hα 6562.8 Å line in a prominence has a Gaussian core set by the Doppler width, which depends on the kinetic temperature and any non-thermal (turbulent) velocity:

$$ \Delta \lambda_D = \frac{\lambda_0}{c} \sqrt{\frac{2 k_B T}{m_H} + \xi^2} $$

Here we synthesize an emission profile for several temperatures with constant non-thermal velocity ξ = 5 km/s.

**한국어.** 프로미넌스의 Hα 6562.8 Å 선의 가우시안 코어는 도플러 폭에 의해 결정되며, 운동 온도와 비열적(난류) 속도에 의존한다. 여기서는 ξ = 5 km/s 고정 하에 여러 온도에서 방출 프로파일을 합성한다.

In [ ]:
def halpha_profile(lam_A, T=T_PROM, xi_kms=5.0, lam0=6562.8, I0=1.0):
    """Synthesize a Gaussian emission Halpha profile.

    Args:
        lam_A: Wavelength array [Angstrom].
        T: Kinetic temperature [K].
        xi_kms: Non-thermal broadening velocity [km/s].
        lam0: Line center wavelength [Angstrom].
        I0: Peak intensity (arbitrary units).

    Returns:
        Tuple (intensity array, Doppler width in Angstrom).
    """
    v_th_cms = np.sqrt(2 * K_B * T / M_H)  # thermal velocity [cm/s]
    xi_cms = xi_kms * 1e5
    v_tot = np.sqrt(v_th_cms**2 + xi_cms**2)
    dlam_D = lam0 * v_tot / C_LIGHT  # Doppler width [A]
    return I0 * np.exp(-((lam_A - lam0) / dlam_D) ** 2), dlam_D


lam = np.linspace(6561.5, 6564.1, 400)
fig, ax = plt.subplots(figsize=(9, 5))
for T in (7000, 8000, 10000, 15000):
    prof, dlam = halpha_profile(lam, T=T, xi_kms=5.0)
    ax.plot(lam, prof, lw=2, label=f'T = {T} K, $\\Delta\\lambda_D$ = {dlam:.3f} Å')
ax.axvline(6562.8, color='k', ls='--', alpha=0.5, label='line center 6562.8 Å')
ax.set_xlabel('Wavelength [Å]')
ax.set_ylabel('Relative intensity')
ax.set_title('Synthetic H$\\alpha$ emission profiles / 합성 Hα 방출 프로파일')
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Formation-Temperature vs Density Phase Diagram / 형성 온도–밀도 상태 다이어그램

**English.** This map places solar atmospheric regions (photosphere, chromosphere, prominence core, PCTR, transition region, quiet-Sun corona, coronal cavity) on a log T–log n phase plane, illustrating the low-T, high-n niche that prominences occupy — approximately two orders of magnitude cooler and two orders of magnitude denser than their ambient corona.

**한국어.** 이 그림은 태양 대기 각 영역(광구, 채층, 프로미넌스 코어, PCTR, 전이영역, 정온 코로나, 코로나 공동)을 log T–log n 상태 평면에 배치하여, 프로미넌스가 주변 코로나보다 약 100배 낮은 온도와 100배 높은 밀도를 가진 특별한 위치에 있음을 보여준다.

In [ ]:
# Regions: (label, log T, log n, color)
regions = [
    ('Photosphere\n광구', 3.78, 17.0, 'tab:orange'),
    ('Chromosphere\n채층', 3.9, 11.5, 'gold'),
    ('Prom. core\n프로미넌스 코어', 3.9, 10.5, 'tab:red'),
    ('PCTR\n전이층', 5.0, 10.0, 'tab:pink'),
    ('Transition region\n전이영역', 5.5, 9.5, 'tab:purple'),
    ('Quiet corona\n정온 코로나', 6.0, 8.5, 'tab:blue'),
    ('Coronal cavity\n코로나 공동', 6.3, 7.8, 'tab:cyan'),
    ('AR filament\n활동영역 필라멘트', 3.9, 11.0, 'tab:brown'),
    ('Hot prom. shroud\n뜨거운 슈라우드', 6.3, 8.0, 'tab:green'),
]

fig, ax = plt.subplots(figsize=(10, 7))
for label, logT, logn, color in regions:
    ax.scatter(logT, logn, s=250, c=color, edgecolor='k', zorder=3)
    ax.annotate(label, (logT, logn), xytext=(8, 8),
                textcoords='offset points', fontsize=9)

# Shaded region highlighting the prominence regime
ax.add_patch(Rectangle((3.8, 9.0), 0.5, 2.5, alpha=0.15,
                       color='tab:red', label='Prominence regime / 프로미넌스 영역'))

ax.set_xlabel('log T [K]  (formation temperature / 형성 온도)')
ax.set_ylabel('log n [cm$^{-3}$]  (electron/hydrogen density)')
ax.set_title('Solar atmosphere phase diagram\n태양 대기 상태 다이어그램')
ax.set_xlim(3.5, 6.7)
ax.set_ylim(7, 18)
ax.grid(alpha=0.3)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 4. Filament Channel Magnetic Dip / 필라멘트 채널 자기 dip

**English.** A *dip* is a concave-up segment of a magnetic field line where plasma can accumulate. This visualization shows field lines in a 2D sheared-arcade filament channel, with a dip visible above the PIL. We construct a simple 2D analytic arcade field and superpose a localized dip perturbation to mimic the plasma-loading deformation.

**한국어.** *dip*은 자기력선의 위로 오목한 부분으로, 플라즈마가 축적될 수 있다. 이 시각화는 2D sheared-arcade 필라멘트 채널의 자기력선을 보여주며, PIL 위에 dip이 보인다. 아케이드 기하에 국소 dip 섭동을 중첩하여 플라즈마 부하에 의한 변형을 모방한다.

In [ ]:
def arcade_field(x, z, L=3.0e9, B0=10.0):
    """2D arcade magnetic field with a localized dip perturbation.

    A simple potential arcade: B_x = B0 cos(pi x/L) exp(-pi z/L),
                               B_z = -B0 sin(pi x/L) exp(-pi z/L),
    with a Gaussian dip perturbation added to B_z to mimic the
    concave-up deformation created by plasma weight.

    Args:
        x: Horizontal coordinate [cm], perpendicular to PIL.
        z: Vertical coordinate [cm].
        L: Arcade half-width [cm] (3e9 cm = 30 Mm).
        B0: Field strength [G].

    Returns:
        (Bx, Bz) in gauss at each (x, z).
    """
    decay = np.exp(-np.pi * z / L)
    Bx = B0 * np.cos(np.pi * x / L) * decay
    Bz = -B0 * np.sin(np.pi * x / L) * decay
    # Localized dip perturbation centered at (0, 0.15L)
    dip = 0.2 * B0 * np.exp(-((x / (0.15 * L))**2 +
                              ((z - 0.15 * L) / (0.10 * L))**2))
    return Bx, Bz + dip


L_cm = 3.0e9  # 30 Mm arcade
x = np.linspace(-L_cm, L_cm, 200)
z = np.linspace(0.01 * L_cm, 1.0 * L_cm, 200)
X, Z = np.meshgrid(x, z)
Bx, Bz = arcade_field(X, Z, L=L_cm)
B_mag = np.sqrt(Bx**2 + Bz**2)

fig, ax = plt.subplots(figsize=(10, 7))
strm = ax.streamplot(X / 1e8, Z / 1e8, Bx, Bz, density=1.8,
                     color=np.log10(B_mag + 0.01), cmap='viridis', linewidth=0.9)
plt.colorbar(strm.lines, ax=ax, label='log$_{10}$ |B| [G]')

ax.axvline(0, color='r', ls='--', lw=1.5, alpha=0.7, label='PIL / 극성 반전선')
ax.scatter(0, 0.15 * L_cm / 1e8, s=400, c='red', marker='*',
           edgecolor='k', zorder=5, label='Prominence / 프로미넌스')

ax.set_xlabel('x [Mm]  (perpendicular to PIL / PIL 수직)')
ax.set_ylabel('z [Mm]  (height above photosphere / 광구 위 고도)')
ax.set_title('2D sheared arcade with prominence dip\n2D 전단 아케이드와 프로미넌스 dip')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Radiative Cooling Rate vs Temperature / 복사 냉각률–온도 곡선

**English.** The radiative loss function Λ(T) characterizes the energy lost per unit volume per unit time by an optically thin plasma:

$$ \mathcal{L}_{\rm rad} = n_e n_H \Lambda(T) $$

For prominence-to-corona temperatures (10^4–10^7 K), Λ(T) peaks around log T = 5.0–5.3 (the radiative instability regime). We use the Rosner–Tucker–Vaiana (1978) piecewise approximation.

**한국어.** 복사 손실 함수 Λ(T)는 광학적으로 얇은 플라즈마가 단위 부피·시간당 잃는 에너지를 특징짓는다: $\mathcal{L}_{\rm rad} = n_e n_H \Lambda(T)$. 프로미넌스–코로나 온도(10^4–10^7 K)에서 Λ(T)는 log T = 5.0–5.3 부근에서 최대(복사 불안정성 영역). Rosner–Tucker–Vaiana 근사로 그린다.

In [ ]:
def rtv_cooling(T):
    """Rosner-Tucker-Vaiana (1978) piecewise radiative loss function.

    Args:
        T: Temperature [K] (array-like).

    Returns:
        Lambda(T) [erg cm^3 s^-1] evaluated element-wise.
    """
    T = np.asarray(T, dtype=float)
    logT = np.log10(T)
    Lam = np.zeros_like(T)
    for i, lt in np.ndenumerate(logT):
        if lt < 4.97:
            Lam[i] = 10**(-21.94)
        elif lt < 5.67:
            Lam[i] = 10**(-17.73) * T[i]**(-2.0 / 3.0)
        elif lt < 6.18:
            Lam[i] = 10**(-21.94)
        elif lt < 6.55:
            Lam[i] = 10**(-10.4) * T[i]**(-2.0)
        elif lt < 6.90:
            Lam[i] = 10**(-21.94)
        elif lt < 7.63:
            Lam[i] = 10**(-17.73) * T[i]**(-2.0 / 3.0)
        else:
            Lam[i] = 10**(-26.6) * T[i]**(1.0 / 4.0)
    return Lam


T_arr = np.logspace(4.0, 7.5, 400)
Lam = rtv_cooling(T_arr)

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(T_arr, Lam, 'b-', lw=2)

ax.axvspan(7e3, 2e4, alpha=0.2, color='red',
           label='Prominence core / 프로미넌스 코어 (7–20 kK)')
ax.axvspan(1e5, 2e5, alpha=0.2, color='orange',
           label='PCTR / 전이층 (log T ≈ 5.0–5.3, DEM min)')
ax.axvspan(1e6, 3e6, alpha=0.2, color='blue',
           label='Quiet corona / 정온 코로나 (1–3 MK)')

ax.set_xlabel('Temperature T [K]')
ax.set_ylabel('Λ(T) [erg cm$^3$ s$^{-1}$]')
ax.set_title('Optically-thin radiative loss function (Rosner et al. 1978)\n광학적 얇은 복사 손실 함수')
ax.legend(loc='upper right')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Estimate prominence radiative loss rate / 프로미넌스 복사 손실률 추정
T_core = T_PROM
n_core = N_PROM
Lam_core = rtv_cooling(np.array([T_core]))[0]
L_rad = n_core * n_core * Lam_core
print('\nProminence radiative loss rate / 프로미넌스 복사 손실률:')
print(f'  T = {T_core:.0f} K, n_e = {n_core:.1e} cm^-3')
print(f'  Λ(T) = {Lam_core:.2e} erg cm^3 s^-1')
print(f'  L_rad = n_e n_H Λ = {L_rad:.2e} erg cm^-3 s^-1')
print(f'  Over 1 Mm column: {L_rad * 1e8:.2e} erg cm^-2 s^-1')
print('\n=> Steady radiation requires an unknown heating source (Parenti 2014, §1).')

## 6. Summary / 요약

**English.** This notebook illustrated five key aspects of solar prominences from Parenti (2014):
1. **Kippenhahn–Schlüter support**: the pressure scale height H_p ≈ 345 km sets the thickness of the sheet; magnetic tension from a ~10 G field can support cool, dense plasma.
2. **Hα profile**: at T = 8000 K the Doppler width is ~0.3 Å — much narrower than typical active-region lines — reflecting the cool prominence core.
3. **Phase diagram**: prominences occupy a unique log T ≈ 3.9, log n ≈ 10.5 corner of atmosphere space, ~100× cooler and ~100× denser than ambient corona.
4. **Magnetic dip**: a 2D sheared arcade produces the characteristic dipped field lines that host prominence plasma along the PIL.
5. **Radiative cooling**: prominence core loses ~10^-9 erg cm^-3 s^-1 — requiring continuous heating to maintain steady state, one of the unresolved puzzles.

**한국어.** 본 노트북은 Parenti (2014)에서 다룬 태양 프로미넌스의 다섯 가지 핵심 측면을 시각화했다:
1. **KS 지지**: 압력 스케일 높이 H_p ≈ 345 km이 sheet 두께를 설정; ~10 G 자장의 자기 장력이 저온 고밀도 플라즈마를 지지한다.
2. **Hα 프로파일**: T = 8000 K에서 도플러 폭 ~0.3 Å — 전형적 활동영역 선보다 훨씬 좁음 — 이 저온 프로미넌스 코어를 반영.
3. **상태 다이어그램**: 프로미넌스는 log T ≈ 3.9, log n ≈ 10.5라는 대기 공간의 독특한 모서리를 차지하며, 주변 코로나보다 ~100배 차갑고 ~100배 밀도가 높다.
4. **자기 dip**: 2D 전단 아케이드는 PIL을 따라 프로미넌스 플라즈마를 담는 특징적인 dip 자기력선을 생성.
5. **복사 냉각**: 프로미넌스 코어는 ~10^-9 erg cm^-3 s^-1을 잃으며, 정상상태 유지를 위해 지속적 가열이 필요하다 — 미해결 문제 중 하나.